# Diffusion-TS
This notebook contains the training of the base Diffusionmodels (Phase 1). Which are the used in the forcasting procedure.

## Drive Mounten und notwendige Packete


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install ema-pytorch==0.2.1;pandas==1.5.0;scikit-learn==1.1.2;scipy==1.8.1;seaborn==0.12.2;tqdm==4.64.1;dm-control==1.0.12;dm-env==1.6;dm-tree==0.1.8;mujoco==2.3.4;gluonts==0.12.6;pyyaml==6.0
import os
import torch
import numpy as np
import pandas as pd
import sys
from sklearn.preprocessing import MinMaxScaler
sys.path.insert(0,'/content/drive/MyDrive/Masterthesis')
from engine.solver import Trainer
from Utils.metric_utils import visualization
from Data.build_dataloader import build_dataloader
from Utils.io_utils import load_yaml_config, instantiate_from_config
from Models.interpretable_diffusion.model_utils import unnormalize_to_zero_to_one

from Utils.metric_utils import display_scores
from Utils.discriminative_metric import discriminative_score_metrics
from Utils.predictive_metric import predictive_score_metrics

import itertools
import yaml
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt


from Utils.cross_correlation import CrossCorrelLoss

## Grid-Search Blue-Chip


In [ ]:
#Set the length of the training data
bluechip_train_length = 4564
config_path_bluechip = '/content/drive/MyDrive/Masterthesis/Config/bluechip_sp500.yaml'
#current setting
train_length = bluechip_train_length
config =config_path_bluechip
#Set up the Grid-Search parameters
hyper_grid = {
    'training.batch_size': [64,128],
    'model.params.n_heads': [4,8],
    'model.params.d_model': [64, 128]
}

def set_nested(config, key, value):
    parts = key.split('.')
    d = config
    for p in parts[:-1]:
        d = d.setdefault(p, {})
    d[parts[-1]] = value
#this functions slices the data into overlapping sequences
def Sequence_slicer(data, window):
    num_variables = data.shape[1]
    seq_length = window
    overlap_size = window - 1
    num_points = data.shape[0]
    step_size = seq_length - overlap_size
    num_sequences = (num_points - seq_length) // step_size + 1
    trimmed_data = np.array([data[i:i + seq_length] for i in range(0, num_points - seq_length + 1, step_size)])

    return trimmed_data.reshape(num_sequences, seq_length, num_variables)

if __name__ == '__main__':
    base_config_path = config
    base_save_root = '/content/drive/MyDrive/Masterthesis/grid_search/blueChip'
    os.makedirs(base_save_root, exist_ok=True)
    #we will iterate 5 times thrue the tests
    iterations = 5

    keys, values = zip(*hyper_grid.items())
    for vals in itertools.product(*values):
        #load and change the configs depending on the Grid-Search parameters
        cfg = load_yaml_config(base_config_path)
        name_parts = []
        for k, v in zip(keys, vals):
            set_nested(cfg, k, v)
            part = f"{k.split('.')[-1]}{v}"
            name_parts.append(part)
        run_name = '_'.join(name_parts)

        save_dir = os.path.join(base_save_root, run_name)
        os.makedirs(save_dir, exist_ok=True)


        cfg.setdefault('solver', {})
        cfg['solver']['results_folder'] = save_dir

        class Args: pass
        args = Args()
        args.config_path = base_config_path
        args.save_dir = save_dir
        args.gpu = 0

        print(cfg) #double check if correct parameters ar set
        device = torch.device(f'cuda:{args.gpu}' if torch.cuda.is_available() else 'cpu')

        dl_info = build_dataloader(cfg, args)
        model = instantiate_from_config(cfg['model']).to(device)
        #load the model
        trainer = Trainer(config=cfg, args=args, model=model, dataloader=dl_info)

        #trainer.load(milestone = 10) #train from a given checkpoint
        trainer.train() #train the model

        dataset = dl_info['dataset']
        seq_length, feature_dim = dataset.window, dataset.var_num
        ori_path = cfg['dataloader']['train_dataset']['params']['data_root']
        initial_ori_data = pd.read_csv(ori_path).values[:train_length,:]

        scaler = MinMaxScaler()
        scaled_ori_data = scaler.fit_transform(initial_ori_data)

        reshaped_ori_data = Sequence_slicer(scaled_ori_data, seq_length)
        # for the tests/metrics we need a sample of generated data
        fake = trainer.sample(num=len(reshaped_ori_data), size_every=len(reshaped_ori_data) +1, shape=[seq_length, feature_dim])
        if dataset.auto_norm:
            fake = unnormalize_to_zero_to_one(fake)
        np.save(os.path.join(save_dir, 'ddpm_fake.npy'), fake)
        # initialize the scores arrays
        discriminative_scores = []
        predictive_scores = []
        correlational_scores = []

        pdf_path = os.path.join(save_dir, 'metrics_analysis.pdf')
        with PdfPages(pdf_path) as pdf:
            num_samples_to_compare = min(len(reshaped_ori_data), len(fake))
            #viszual methods
            for method in ['pca', 'tsne', 'kernel']:
                visualization(ori_data=reshaped_ori_data, generated_data=fake, analysis=method, compare=num_samples_to_compare, pdf = pdf)
            #Discriminative Score
            for i in range(iterations):
                temp_disc, fake_acc, real_acc = discriminative_score_metrics(reshaped_ori_data[:num_samples_to_compare], fake[:num_samples_to_compare])
                discriminative_scores.append(temp_disc)
                print(f'Iter {i}: Discriminative Score = {temp_disc}, Fake Acc = {fake_acc}, Real Acc = {real_acc}')
            #Predictive Score
            for i in range(iterations):
                temp_pred = predictive_score_metrics(reshaped_ori_data[:num_samples_to_compare], fake[:num_samples_to_compare])
                predictive_scores.append(temp_pred)
                print(f'Iter {i}: Predictive MAE = {temp_pred}')

            def random_choice(size, num_select):
                return np.random.randint(low=0, high=size, size=(num_select,))

            x_real = torch.from_numpy(reshaped_ori_data)
            x_fake = torch.from_numpy(fake)
            for i in range(iterations):
                current_num_select = min(num_samples_to_compare, x_real.shape[0], x_fake.shape[0])
                if current_num_select == 0:
                    correlational_scores.append(np.nan)
                    continue
                #Correlation Score
                real_idx = random_choice(x_real.shape[0], num_select=current_num_select)
                fake_idx = random_choice(x_fake.shape[0], num_select=current_num_select)
                corr = CrossCorrelLoss(x_real[real_idx], name='CrossCorrelLoss')
                loss = corr.compute(x_fake[fake_idx])
                correlational_scores.append(loss.item())
                print(f'Iter {i}: Cross-Corr Loss = {loss.item()}')

            # Summarize the the metrics
            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.axis('off')
            summary_text = "Zusammenfassung der Metriken:\n\n"

            for i in range(iterations):
                summary_text += f"Iteration {i+1}:\n"
                summary_text += f"  - Discriminative Score: {discriminative_scores[i]:.4f}\n"
                summary_text += f"  - Predictive MAE: {predictive_scores[i]:.4f}\n"
                if i < len(correlational_scores) and not np.isnan(correlational_scores[i]):
                     summary_text += f"  - Cross-Corr Loss: {correlational_scores[i]:.4f}\n\n"
                else:
                    summary_text += f"  - Cross-Corr Loss: N/A\n\n"

            mean_disc = np.mean(discriminative_scores) if discriminative_scores else np.nan
            mean_pred = np.mean(predictive_scores) if predictive_scores else np.nan

            valid_corr_scores = [score for score in correlational_scores if not np.isnan(score)]
            mean_corr = np.mean(valid_corr_scores) if valid_corr_scores else np.nan

            summary_text += "-" * 40 + "\n"
            summary_text += "Mittelwerte über alle Iterationen:\n\n"
            summary_text += f"  - Mean Discriminative Score: {mean_disc:.4f}\n"
            summary_text += f"  - Mean Predictive MAE: {mean_pred:.4f}\n"

            if not np.isnan(mean_corr):
                summary_text += f"  - Mean Cross-Corr Loss: {mean_corr:.4f}\n"
            else:
                summary_text += f"  - Mean Cross-Corr Loss: N/A\n"

            ax.text(0.05, 0.95, summary_text, verticalalignment='top', fontsize=10)
            pdf.savefig()
            plt.close()

        print(f"Finished run {run_name}, metrics saved in {pdf_path}")

## CrossAsset
Grid-Search procedure just like above for the Cross-Asset portfolio.

In [ ]:

crossAsset_train_length = 5298
train_length = crossAsset_train_length
config_path_crossAsset = '/content/drive/MyDrive/Masterthesis/Config/crossAsset.yaml'

config =config_path_crossAsset

hyper_grid = {
    'training.batch_size': [64,128],
    'model.params.n_heads': [4,8],
    'model.params.d_model': [64,128]
}

def set_nested(config, key, value):
    parts = key.split('.')
    d = config
    for p in parts[:-1]:
        d = d.setdefault(p, {})
    d[parts[-1]] = value

def Sequence_slicer(data, window):
    num_variables = data.shape[1]
    seq_length = window
    overlap_size = window - 1
    num_points = data.shape[0]
    step_size = seq_length - overlap_size
    num_sequences = (num_points - seq_length) // step_size + 1

    trimmed_data = np.array([data[i:i + seq_length] for i in range(0, num_points - seq_length + 1, step_size)])

    return trimmed_data.reshape(num_sequences, seq_length, num_variables)

if __name__ == '__main__':
    base_config_path = config
    base_save_root = '/content/drive/MyDrive/Masterthesis/grid_search/crossAsset'
    os.makedirs(base_save_root, exist_ok=True)

    iterations = 5

    keys, values = zip(*hyper_grid.items())
    for vals in itertools.product(*values):

        cfg = load_yaml_config(base_config_path)
        name_parts = []
        for k, v in zip(keys, vals):
            set_nested(cfg, k, v)
            part = f"{k.split('.')[-1]}{v}"
            name_parts.append(part)
        run_name = '_'.join(name_parts)

        save_dir = os.path.join(base_save_root, run_name)
        os.makedirs(save_dir, exist_ok=True)


        cfg.setdefault('solver', {})
        cfg['solver']['results_folder'] = save_dir

        class Args: pass
        args = Args()
        args.config_path = base_config_path
        args.save_dir = save_dir
        args.gpu = 0

        device = torch.device(f'cuda:{args.gpu}' if torch.cuda.is_available() else 'cpu')
        print(cfg)
        dl_info = build_dataloader(cfg, args)
        model = instantiate_from_config(cfg['model']).to(device)
        trainer = Trainer(config=cfg, args=args, model=model, dataloader=dl_info)

        #trainer.load(milestone = 10)
        trainer.train()

        dataset = dl_info['dataset']
        seq_length, feature_dim = dataset.window, dataset.var_num
        ori_path = cfg['dataloader']['train_dataset']['params']['data_root']

        initial_ori_data = pd.read_csv(ori_path).values[:train_length,:]

        scaler = MinMaxScaler()

        scaled_ori_data = scaler.fit_transform(initial_ori_data)


        reshaped_ori_data = Sequence_slicer(scaled_ori_data, seq_length)

        fake = trainer.sample(num=len(reshaped_ori_data), size_every=len(reshaped_ori_data) +1, shape=[seq_length, feature_dim])
        if dataset.auto_norm:
            fake = unnormalize_to_zero_to_one(fake)
        np.save(os.path.join(save_dir, 'ddpm_fake.npy'), fake)


        discriminative_scores = []
        predictive_scores = []
        correlational_scores = []

        pdf_path = os.path.join(save_dir, 'metrics_analysis.pdf')
        with PdfPages(pdf_path) as pdf:

            num_samples_to_compare = min(len(reshaped_ori_data), len(fake))
            print(f"Comparing {num_samples_to_compare} samples for metrics.")


            for method in ['pca', 'tsne', 'kernel']:
                visualization(ori_data=reshaped_ori_data, generated_data=fake, analysis=method, compare=num_samples_to_compare, pdf = pdf)


            for i in range(iterations):
                temp_disc, fake_acc, real_acc = discriminative_score_metrics(reshaped_ori_data[:num_samples_to_compare], fake[:num_samples_to_compare])
                discriminative_scores.append(temp_disc)
                print(f'Iter {i}: Discriminative Score = {temp_disc}, Fake Acc = {fake_acc}, Real Acc = {real_acc}')


            for i in range(iterations):
                temp_pred = predictive_score_metrics(reshaped_ori_data[:num_samples_to_compare], fake[:num_samples_to_compare])
                predictive_scores.append(temp_pred)
                print(f'Iter {i}: Predictive MAE = {temp_pred}')


            def random_choice(size, num_select):
                return np.random.randint(low=0, high=size, size=(num_select,))

            x_real = torch.from_numpy(reshaped_ori_data)
            x_fake = torch.from_numpy(fake)
            for i in range(iterations):
                current_num_select = min(num_samples_to_compare, x_real.shape[0], x_fake.shape[0])
                if current_num_select == 0:
                    print("Warning: No samples available for cross-correlational score calculation.")
                    correlational_scores.append(np.nan)
                    continue

                real_idx = random_choice(x_real.shape[0], num_select=current_num_select)
                fake_idx = random_choice(x_fake.shape[0], num_select=current_num_select)
                corr = CrossCorrelLoss(x_real[real_idx], name='CrossCorrelLoss')
                loss = corr.compute(x_fake[fake_idx])
                correlational_scores.append(loss.item())
                print(f'Iter {i}: Cross-Corr Loss = {loss.item()}')


            fig, ax = plt.subplots(figsize=(8.27, 11.69))
            ax.axis('off')
            summary_text = "Zusammenfassung der Metriken:\n\n"

            for i in range(iterations):
                summary_text += f"Iteration {i+1}:\n"
                summary_text += f"  - Discriminative Score: {discriminative_scores[i]:.4f}\n"
                summary_text += f"  - Predictive MAE: {predictive_scores[i]:.4f}\n"
                if i < len(correlational_scores) and not np.isnan(correlational_scores[i]):
                     summary_text += f"  - Cross-Corr Loss: {correlational_scores[i]:.4f}\n\n"
                else:
                    summary_text += f"  - Cross-Corr Loss: N/A\n\n"

            mean_disc = np.mean(discriminative_scores) if discriminative_scores else np.nan
            mean_pred = np.mean(predictive_scores) if predictive_scores else np.nan

            valid_corr_scores = [score for score in correlational_scores if not np.isnan(score)]
            mean_corr = np.mean(valid_corr_scores) if valid_corr_scores else np.nan

            summary_text += "-" * 40 + "\n"
            summary_text += "Mittelwerte über alle Iterationen:\n\n"
            summary_text += f"  - Mean Discriminative Score: {mean_disc:.4f}\n"
            summary_text += f"  - Mean Predictive MAE: {mean_pred:.4f}\n"

            if not np.isnan(mean_corr):
                summary_text += f"  - Mean Cross-Corr Loss: {mean_corr:.4f}\n"
            else:
                summary_text += f"  - Mean Cross-Corr Loss: N/A\n"


            ax.text(0.05, 0.95, summary_text, verticalalignment='top', fontsize=10)
            pdf.savefig()
            plt.close()

        print(f"Finished run {run_name}, metrics saved in {pdf_path}")